# Diag outliers

Find:
- Ensemble members / GWLs where the mean difference in temperature between GWLs is not in line with the GWL difference
- Ensemble members / GWLs where the mean / std / extremes are outliers

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
import glob
import zarr
import functools
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

from funcs_support import get_filepaths,get_params,utility_save
from funcs_aux import _find_main_variable, _load_gwls, get_landmask
dir_list = get_params()

In [2]:
from distributed import Client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39545 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/crhtc75.hpc.ucar.edu/11423/proxy/39545/status,
Dashboard: /node/crhtc75.hpc.ucar.edu/11423/proxy/39545/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44809,Workers: 0
Dashboard: /node/crhtc75.hpc.ucar.edu/11423/proxy/39545/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45271,Total threads: 3
Dashboard: /node/crhtc75.hpc.ucar.edu/11423/proxy/46323/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:41145,


In [3]:
fns_bcd = glob.glob('/glade/campaign/cgd/cas/schwarzwald/bcd_me/*.zarr')
fns_bcd = np.sort(fns_bcd)

fns_bcd_df = pd.DataFrame({'path':fns_bcd})
fns_bcd_df['src'] = [re.split(r'\_',fn)[-2] for fn in fns_bcd_df.path.values]
fns_bcd_df['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in fns_bcd_df.path.values]
fns_bcd_df = fns_bcd_df.set_index(['src','model'])

## Differentiate wrt GWL
To catch ensemble members that aren't changing as expected

In [ ]:
for row in fns_bcd_df.loc['qdm'].iterrows():
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore')
        ds = xr.open_zarr(row[1]['path'],consolidated=False,decode_timedelta=False)

    ds = ds.chunk({'gwl':-1})
    ds_dgwl = ds.differentiate('gwl').weighted(np.cos(np.deg2rad(ds.lat))).mean(('lat','lon','dayofyear','year'))

    output_fn = f'{dir_list['aux']}diagnostics/diag_qdm_gwldiffmean_{row[0]}.nc'
    ds_dgwl.attrs = {'SOURCE':'diag_outliers.ipynb',
                     'DESCRIPTION':'Near-global mean values, differentiated wrt GWL. Should be close to 1.'}

    ds_dgwl = ds_dgwl.compute()

    utility_save(ds_dgwl,output_fn)

/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_gwldiffmean_ACCESS-CM2.nc removed to allow overwrite!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_gwldiffmean_ACCESS-CM2.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_gwldiffmean_ACCESS-ESM1-5.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_gwldiffmean_BCC-CSM2-MR.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_gwldiffmean_BCC-ESM1.nc saved!


## Global summary statistics 
To identify outliers

In [4]:
fns_bcd = glob.glob('/glade/campaign/cgd/cas/schwarzwald/bcd_me/*.zarr')
fns_bcd = np.sort(fns_bcd)

fns_bcd_df = pd.DataFrame({'path':fns_bcd})
fns_bcd_df['src'] = [re.split(r'\_',fn)[-2] for fn in fns_bcd_df.path.values]
fns_bcd_df['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in fns_bcd_df.path.values]
fns_bcd_df = fns_bcd_df.set_index(['src','model'])

In [ ]:
overwrite = False
for row in fns_bcd_df.iterrows():
    output_fn = f'{dir_list['aux']}diagnostics/diag_{row[0][0]}_summstats_{row[0][1]}.nc'
    if overwrite or not os.path.exists(output_fn):
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            ds = xr.open_zarr(row[1]['path'],consolidated=False,decode_timedelta=False)
            ds = ds[[v for v in ds]] # Occasional issue with stats being calculated over has_data, even though it's a coordinate
            
            dsw = ds.weighted(np.cos(np.deg2rad(ds.lat)))

            dims_tmp = np.unique(np.array([[k for k in ds[d].sizes] for d in ds]).flatten())
            dimlist = [d for d in dims_tmp if d not in ['gwl','binC','binF','degree','proj_base','idv','bnds','variable']]
    
            ds_stats = xr.concat([dsw.mean(dimlist).expand_dims({'typ':['mean']}),
                                  dsw.std(dimlist).expand_dims({'typ':['std']}),
                                  ds.max(dimlist).expand_dims({'typ':['max']}),
                                  ds.min(dimlist).expand_dims({'typ':['min']})],
                                  #dsw.quantile(0.1,dim=('lat','lon','dayofyear','year')).expand_dims({'typ':['q10']}),
                                  #dsw.quantile(0.9,dim=('lat','lon','dayofyear','year')).expand_dims({'typ':['q90']})],
                                 dim='typ')
            
           
            ds_stats.attrs = {'SOURCE':'diag_outliers.ipynb',
                             'DESCRIPTION':'Near-global mean, max, min, and standard deviation, across time and (area-weighted) space.'}
    
            ds_stats = ds_stats.compute()
            
            utility_save(ds_stats,output_fn)
    else:
        print(output_fn+' exists, skipped!')

/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_CESM2-WACCM.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_CNRM-CM6-1.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_CNRM-ESM2-1.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_CanESM5.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_EC-Earth3-Veg.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_EC-Earth3.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_FGOALS-g3.nc exists, skipped!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm-byyr_summstats_IPSL-CM6A-LR.nc exists, skipped!
/glade

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/dask/array/reductions.py:294: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/dask/array/reductions.py:327: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(x_chunk, axis=axis, keepdims=keepdims)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/dask/array/reductions.py:327: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(x_chunk, axis=axis, keepdims=keepdims)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/dask/array/reductions.py:294: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/dask/array/reductions.py:294: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis,

/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_summstats_IPSL-CM6A-LR.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_summstats_MIROC-ES2L.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_summstats_MIROC6.nc saved!
/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_summstats_MPI-ESM-1-2-HAM.nc saved!


In [15]:
dsw.mean(dimlist).expand_dims({'typ':['mean']}).compute()

<xarray.Dataset> Size: 19kB
Dimensions:                         (typ: 1, idv: 25, gwl: 8, proj_base: 5)
Coordinates:
  * typ                             (typ) object 8B 'mean'
    run                             (idv) <U9 900B 'r10i1p1f1' ... 'r3i1p1f1'
    experiment                      (idv) <U6 600B 'ssp245' ... 'ssp585'
    model                           (idv) <U12 1kB 'IPSL-CM6A-LR' ... 'IPSL-C...
  * gwl                             (gwl) float64 64B 0.61 1.0 1.5 ... 3.5 4.0
    flag_has_historical_duplicates  (idv, gwl) bool 200B True True ... False
  * proj_base                       (proj_base) <U6 120B 'ERA5' ... 'PRISM'
Dimensions without coordinates: idv
Data variables:
    tas                             (typ, idv, proj_base, gwl) float64 8kB na...
    tasmax                          (typ, idv, proj_base, gwl) float64 8kB 29...
Attributes:
    DESCRIPTION:  IPSL-CM6A-LR ScenarioMIP runs, bias-corrected using Quantil...
    Project:      Bias-Corrected, Downscaled Massive Ensemble (BCD-ME)
    Creators:     Kevin Schwarzwald, Nathan Lenssen, Radley Horton, and Gerno...
    Version:      v1.0
    License:      CC BY 4.0 (for proj_base == ERA5, MERRA-2, or GMFD), CC BY-...

In [8]:
ds_stats

<xarray.Dataset> Size: 68kB
Dimensions:                         (typ: 4, idv: 25, proj_base: 5, gwl: 8,
                                     variable: 2)
Coordinates:
  * typ                             (typ) object 32B 'mean' 'std' 'max' 'min'
    run                             (idv) <U9 900B 'r10i1p1f1' ... 'r3i1p1f1'
    experiment                      (idv) <U6 600B 'ssp245' ... 'ssp585'
    model                           (idv) <U12 1kB 'IPSL-CM6A-LR' ... 'IPSL-C...
  * proj_base                       (proj_base) <U6 120B 'ERA5' ... 'PRISM'
  * gwl                             (gwl) float64 64B 0.61 1.0 1.5 ... 3.5 4.0
    flag_has_historical_duplicates  (idv, gwl) bool 200B dask.array<chunksize=(25, 8), meta=np.ndarray>
    flag_has_data                   (variable, idv, gwl) bool 400B dask.array<chunksize=(2, 25, 8), meta=np.ndarray>
  * variable                        (variable) object 16B 'tas' 'tasmax'
Dimensions without coordinates: idv
Data variables:
    tas                             (typ, idv, proj_base, gwl) float64 32kB dask.array<chunksize=(1, 1, 1, 1), meta=np.ndarray>
    tasmax                          (typ, idv, proj_base, gwl) float64 32kB dask.array<chunksize=(1, 1, 1, 1), meta=np.ndarray>
Attributes:
    SOURCE:       diag_outliers.ipynb
    DESCRIPTION:  Near-global mean, max, min, and standard deviation, across ...

In [30]:
output_fn

'/glade/u/home/schwarzwald/projects/bcd_me_final/aux_data/diagnostics/diag_qdm_summstats_NESM3.nc'